# Live Demo: Gender Wage Gap — ML Analysis

**Course:** AD688 Applied Business Analytics | **Group B**  
**Dataset:** IPUMS USA 2024 ACS + Lightcast Job Postings  
**Research Question:** How much of the gender wage gap is explained by occupation and industry choice?

This notebook demonstrates:
1. Linear Regression Decomposition
2. Random Forest Feature Importance


## Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import sys
import pathlib

root = pathlib.Path.cwd()
while not (root / "_quarto.yml").exists() and root != root.parent:
    root = root.parent

sys.path.insert(0, str(root / "analysis"))

from utils import load_employment, PROCESSED, apply_theme

df = load_employment()

print(f"Loaded {len(df):,} employed workers")
print(f"Columns: {list(df.columns)}")

df[["AGE", "SEX_LABEL", "OCC", "IND", "INCWAGE", "STATE_NAME"]].head()


## Step 1: Context — Gender Gap in IT vs Overall Workforce

IT workforce = Census OCC codes 1000–1999.


In [ ]:
df_it = df[(df["OCC"] >= 1000) & (df["OCC"] <= 1999)]

overall_female_pct = round(df[df["SEX_LABEL"] == "Female"].shape[0] / len(df) * 100, 1)
it_female_pct = round(df_it[df_it["SEX_LABEL"] == "Female"].shape[0] / len(df_it) * 100, 1)

overall_gap = (
    df[df["SEX_LABEL"] == "Male"]["INCWAGE"].mean()
    - df[df["SEX_LABEL"] == "Female"]["INCWAGE"].mean()
)

it_gap = (
    df_it[df_it["SEX_LABEL"] == "Male"]["INCWAGE"].mean()
    - df_it[df_it["SEX_LABEL"] == "Female"]["INCWAGE"].mean()
)

print(f"Overall workforce: {overall_female_pct}% female | Wage gap: ${overall_gap:,.0f}")
print(f"IT workforce:      {it_female_pct}% female | Wage gap: ${it_gap:,.0f}")

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Male",
    x=["Overall", "IT"],
    y=[100 - overall_female_pct, 100 - it_female_pct],
    marker_color="#2c7bb6",
))
fig.add_trace(go.Bar(
    name="Female",
    x=["Overall", "IT"],
    y=[overall_female_pct, it_female_pct],
    marker_color="#d7191c",
))

fig.update_layout(
    barmode="group",
    title="Gender Distribution: Overall vs IT Workforce",
    yaxis_title="%",
    template="plotly_white",
)

apply_theme(fig)
fig.show()


## Step 2: Baseline Regression

Features: age, race, state, and sex.  
This estimates the gender wage gap before controlling for occupation and industry.


In [ ]:
baseline_df = df[["AGE", "RACE_LABEL", "STATE_NAME", "SEX_LABEL", "INCWAGE"]].copy()

baseline_encoded = pd.get_dummies(
    baseline_df,
    columns=["RACE_LABEL", "STATE_NAME", "SEX_LABEL"],
    drop_first=True,
)

X_baseline = baseline_encoded.drop(columns=["INCWAGE"])
y_baseline = baseline_encoded["INCWAGE"]

model_baseline = LinearRegression()
model_baseline.fit(X_baseline, y_baseline)

sex_col = [c for c in X_baseline.columns if "SEX_LABEL" in c][0]
baseline_gender_coef = model_baseline.coef_[list(X_baseline.columns).index(sex_col)]

print(f"Baseline R²: {model_baseline.score(X_baseline, y_baseline):.4f}")
print(f"Raw gender gap controlling for age, race, and state: ${baseline_gender_coef:,.2f}")


## Step 3: Full Regression

Features: age, race, state, sex, occupation group, and industry group.  
This estimates how much of the gap remains after controlling for job type.


In [ ]:
full_df = df[["AGE", "RACE_LABEL", "STATE_NAME", "SEX_LABEL", "OCC", "IND", "INCWAGE"]].copy()

full_df["OCC_GROUP"] = (full_df["OCC"] // 100) * 100
full_df["IND_GROUP"] = (full_df["IND"] // 1000) * 1000

full_encoded = pd.get_dummies(
    full_df.drop(columns=["OCC", "IND"]),
    columns=["RACE_LABEL", "STATE_NAME", "SEX_LABEL", "OCC_GROUP", "IND_GROUP"],
    drop_first=True,
)

X_full = full_encoded.drop(columns=["INCWAGE"])
y_full = full_encoded["INCWAGE"]

model_full = LinearRegression()
model_full.fit(X_full, y_full)

sex_col_full = [c for c in X_full.columns if "SEX_LABEL" in c][0]
full_gender_coef = model_full.coef_[list(X_full.columns).index(sex_col_full)]

explained_pct = ((baseline_gender_coef - full_gender_coef) / baseline_gender_coef) * 100
unexplained_pct = 100 - explained_pct

print(f"Full model R²: {model_full.score(X_full, y_full):.4f}")
print(f"Adjusted gender gap: ${full_gender_coef:,.2f}")
print(f"Explained by occupation/industry: {explained_pct:.1f}%")
print(f"Unexplained gap: {unexplained_pct:.1f}%")


## Step 4: Decomposition Visualization

In [ ]:
explained_amt = baseline_gender_coef - full_gender_coef

fig_wf = go.Figure(go.Waterfall(
    orientation="v",
    measure=["absolute", "relative", "total"],
    x=["Raw Gender Gap", "Explained by<br>Occupation/Industry", "Unexplained Gap"],
    y=[baseline_gender_coef, -explained_amt, 0],
    text=[
        f"${baseline_gender_coef:,.0f}",
        f"-${explained_amt:,.0f}",
        f"${full_gender_coef:,.0f}",
    ],
    textposition="outside",
    connector={"line": {"color": "#888"}},
    decreasing={"marker": {"color": "#1a9641"}},
    increasing={"marker": {"color": "#d7191c"}},
    totals={"marker": {"color": "#2c7bb6"}},
))

fig_wf.update_layout(
    title="Gender Wage Gap Decomposition",
    yaxis_title="Annual Wage Gap ($)",
    showlegend=False,
    template="plotly_white",
)

apply_theme(fig_wf)
fig_wf.show()


## Step 5: Random Forest Feature Importance

Uses a 50,000-row sample for faster live demo execution.


In [ ]:
rf_df = df[["AGE", "RACE_LABEL", "STATE_NAME", "SEX_LABEL", "OCC", "IND", "INCWAGE"]].sample(
    n=50000,
    random_state=42,
).copy()

rf_df["OCC_GROUP"] = (rf_df["OCC"] // 100) * 100
rf_df["IND_GROUP"] = (rf_df["IND"] // 1000) * 1000

rf_encoded = pd.get_dummies(
    rf_df.drop(columns=["OCC", "IND"]),
    columns=["RACE_LABEL", "STATE_NAME", "SEX_LABEL", "OCC_GROUP", "IND_GROUP"],
    drop_first=True,
)

X = rf_encoded.drop(columns=["INCWAGE"])
y = rf_encoded["INCWAGE"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)

print(f"Random Forest trained on {len(X_train):,} workers")
print(f"Train R²: {rf_model.score(X_train, y_train):.4f}")
print(f"Test R²: {rf_model.score(X_test, y_test):.4f}")


## Step 6: Feature Importance — Where Does Gender Rank?

In [ ]:
importances = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

sex_rank = importances[importances["feature"].str.contains("SEX_LABEL")].index[0] + 1

print(f"Gender ranks #{sex_rank} of {len(importances)} features")

occ_path = PROCESSED / "occ_group_labels.csv"

if occ_path.exists():
    occ_labels = pd.read_csv(occ_path)
    occ_lookup = dict(zip(occ_labels["OCC_GROUP"], occ_labels["MAJOR_GROUP"]))
else:
    occ_lookup = {}

def clean_label(f):
    if f == "AGE":
        return "Age"
    if "SEX_LABEL" in f:
        return "Gender (Male)"
    if f.startswith("OCC_GROUP_"):
        grp = int(f.replace("OCC_GROUP_", ""))
        return f"Occupation: {occ_lookup.get(grp, grp)}"
    if f.startswith("IND_GROUP_"):
        return f"Industry Group {f.replace('IND_GROUP_', '')}"
    if f.startswith("STATE_NAME_"):
        return f"State: {f.replace('STATE_NAME_', '')}"
    if f.startswith("RACE_LABEL_"):
        return f"Race: {f.replace('RACE_LABEL_', '')}"
    return f

top15 = importances.head(15).copy()
top15["label"] = top15["feature"].apply(clean_label)
top15["is_gender"] = top15["feature"].str.contains("SEX_LABEL")

fig_lollipop = go.Figure()

fig_lollipop.add_trace(go.Scatter(
    x=top15["importance"],
    y=top15["label"],
    mode="markers",
    marker=dict(
        size=14,
        color=top15["is_gender"].map({True: "#d7191c", False: "#2c7bb6"}),
    ),
))

for _, row in top15.iterrows():
    fig_lollipop.add_shape(
        type="line",
        x0=0,
        x1=row["importance"],
        y0=row["label"],
        y1=row["label"],
        line=dict(
            color="#d7191c" if row["is_gender"] else "#cccccc",
            width=2,
        ),
    )

fig_lollipop.update_layout(
    title=f"Random Forest Feature Importance — Gender Ranks #{sex_rank} of {len(importances)}",
    xaxis_title="Importance",
    yaxis_title="",
    yaxis=dict(categoryorder="total ascending"),
    template="plotly_white",
)

apply_theme(fig_lollipop)
fig_lollipop.show()


## Key Findings

| Finding | Value |
|---|---|
| Raw gender wage gap | ~$24,749/year |
| Adjusted gap after occupation and industry | ~$22,426/year |
| Explained by job segregation | ~9.4% |
| Unexplained gap | ~90.6% |
| Gender rank in Random Forest | #3 of 167 features |

### Interpretation

The gender wage gap is not primarily explained by women choosing lower-paying jobs. Most of the gap persists even after controlling for broad occupation and industry groups. The Random Forest model independently confirms that gender remains one of the strongest predictors of wage in the available feature set.
